# Py-Boost (SketchBoost)

## Что такое Py-Boost?

[Py-Boost](https://github.com/sb-ai-lab/Py-Boost) — GPU-реализация градиентного бустинга от Sber AI Lab с нативной поддержкой **multi-output** задач.

Ключевое отличие от CatBoost/LightGBM/XGBoost:
- **Одна модель** обучается сразу на все таргеты (multi-label / multiclass)
- **SketchBoost** — алгоритм приближённого поиска сплитов для multi-output, который ускоряет обучение в 3-5x без потери качества
- Полностью на GPU (CuPy + Numba), быстрее CatBoost GPU на multi-output задачах

### Ссылки
- GitHub: https://github.com/sb-ai-lab/Py-Boost
- Туториалы: https://github.com/sb-ai-lab/Py-Boost/blob/master/tutorials
- Статья SketchBoost: https://arxiv.org/abs/2112.13054

## Установка

Py-Boost требует:
- **CuPy** (CUDA-совместимый numpy) — версия должна соответствовать CUDA на машине
- **py-boost** — сама библиотека

In [1]:
# Установка зависимостей
# CuPy нужно ставить под вашу версию CUDA (проверьте: nvidia-smi → CUDA Version)
# !pip install cupy-cuda12x   # для CUDA 12.x
# !pip install cupy-cuda11x   # для CUDA 11.x

!pip install -q -U iterative-stratification py-boost

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf 23.8.0 requires cubinlinker, which is not installed.
cudf 23.8.0 requires cupy-cuda11x>=12.0.0, which is not installed.
cudf 23.8.0 requires ptxcompiler, which is not installed.
cuml 23.8.0 requires cupy-cuda11x>=12.0.0, which is not installed.
dask-cudf 23.8.0 requires cupy-cuda11x>=12.0.0, which is not installed.
tensorflow-decision-forests 1.8.1 requires wurlitzer, which is not installed.
apache-beam 2.46.0 requires dill<0.3.2,>=0.3.1.1, but you have dill 0.3.8 which is incompatible.
apache-beam 2.46.0 requires numpy<1.25.0,>=1.14.3, but you have numpy 1.26.4 which is incompatible.
apache-beam 2.46.0 requires protobuf<4,>3.12.2, but you have protobuf 6.33.5 which is incompatible.
apache-beam 2.46.0 requires pyarrow<10.0.0,>=3.0.0, but you have pyarrow 11.0.0 which is incompatible.
cudf 23.8.0 requires cuda

In [2]:
import numpy as np
import pandas as pd
import polars as pl
import cupy as cp
from py_boost import GradientBoosting, SketchBoost
from py_boost.multioutput.sketching import RandomProjectionSketch, RandomSamplingSketch

# Дополнительные стратегии (если установлен cuml):
# from py_boost.multioutput.sketching import TopOutputsSketch, SVDSketch

from sklearn.metrics import roc_auc_score
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
import gc
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = '/kaggle/input/datasets/d1ffic00lt/data-fusion-2026-case-2/'
N_FOLDS = 5
SEED = 42

# Проверяем GPU
print(f'CuPy version: {cp.__version__}')
print(f'CUDA device: {cp.cuda.runtime.getDeviceProperties(0)["name"]}')
print(f'GPU memory: {cp.cuda.Device(0).mem_info[1] / 1e9:.1f} GB')

/opt/conda/lib/python3.10/site-packages/dask/dataframe/_pyarrow_compat.py:23: UserWarning: You are using pyarrow version 11.0.0 which is known to be insecure. See https://www.cve.org/CVERecord?id=CVE-2023-47248 for further details. Please upgrade to pyarrow>=14.0.1 or install pyarrow-hotfix to patch your current version.
  warnings.warn(


CuPy version: 13.0.0
CUDA device: b'Tesla T4'
GPU memory: 15.6 GB


In [3]:
# Только так завелось обучение на Kaggle
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

## 1. Загрузка данных

In [4]:
train_main = pl.read_parquet(f'{DATA_DIR}train_main_features.parquet')
test_main = pl.read_parquet(f'{DATA_DIR}test_main_features.parquet')
train_target = pl.read_parquet(f'{DATA_DIR}train_target.parquet')

target_cols = [c for c in train_target.columns if c != 'customer_id']
feature_cols = [c for c in train_main.columns if c != 'customer_id']

print(f'Features: {len(feature_cols)}, Targets: {len(target_cols)}')
print(f'Train: {train_main.shape[0]}, Test: {test_main.shape[0]}')

Features: 199, Targets: 41
Train: 750000, Test: 250000


In [5]:
# # Опционально (скор падает)
# # Удаляем 26 фичей: 18 с >95% пропусков, 8 константных, 3 дубликата по корреляции
# drop_cols = [
#     'num_feature_106', 'num_feature_110', 'num_feature_112', 'num_feature_118',
#     'num_feature_13',  'num_feature_14',  'num_feature_19',  'num_feature_22',
#     'num_feature_28',  'num_feature_32',  'num_feature_34',  'num_feature_4',
#     'num_feature_43',  'num_feature_44',  'num_feature_45',  'num_feature_46',
#     'num_feature_47',  'num_feature_54',  'num_feature_59',  'num_feature_64',
#     'num_feature_70',  'num_feature_74',  'num_feature_75',  'num_feature_80',
#     'num_feature_81',  'num_feature_9',
# ]

# feature_cols = [c for c in feature_cols if c not in drop_cols]
# print(f'After drop: {len(feature_cols)} features (removed {len(drop_cols)})')

In [6]:
# Py-Boost принимает только numpy float32
# NaN-ы поддерживаются нативно (как CatBoost nan_mode='Min')
X = train_main.select(feature_cols).to_numpy().astype(np.float32)
y = train_target.select(target_cols).to_numpy().astype(np.float32)
X_test = test_main.select(feature_cols).to_numpy().astype(np.float32)

customer_ids_train = train_target['customer_id'].to_numpy()
customer_ids_test = test_main['customer_id'].to_numpy()

del train_main, test_main, train_target
gc.collect()

print(f'X: {X.shape}, y: {y.shape}, X_test: {X_test.shape}')

X: (750000, 199), y: (750000, 41), X_test: (250000, 199)


## 2. Параметры SketchBoost

### Отличия SketchBoost от GradientBoosting

- `SketchBoost` — обёртка над `GradientBoosting` со встроенным `RandomProjectionSketch(1)` и `use_hess=False`
- Эквивалентно:
```python
# Это одно и то же:
model = SketchBoost('bce', ntrees=5000)
model = GradientBoosting('bce', ntrees=5000, 
                         multioutput_sketch=RandomProjectionSketch(1), 
                         use_hess=False)
```

In [7]:
# Базовые параметры — хорошая отправная точка
PARAMS = dict(
    ntrees=5000,
    lr=0.05,              # learning rate
    verbose=200,
    es=200,               # early stopping patience
    lambda_l2=1,          # L2-регуляризация
    subsample=0.8,        # доля строк
    colsample=0.8,        # доля фичей
    min_data_in_leaf=10,  # мин. сэмплов в листе
    max_depth=6,          # глубина дерева
    max_bin=256,          # число бинов квантизации
    gd_steps=1,           # шаги GD в листьях
    use_hess=False,       # SketchBoost по умолчанию без Гессиана
)

print('SketchBoost params:')
for k, v in PARAMS.items():
    print(f'  {k}: {v}')

SketchBoost params:
  ntrees: 5000
  lr: 0.05
  verbose: 200
  es: 200
  lambda_l2: 1
  subsample: 0.8
  colsample: 0.8
  min_data_in_leaf: 10
  max_depth: 6
  max_bin: 256
  gd_steps: 1
  use_hess: False


## 3. CV с SketchBoost

Ключевые моменты:
- `SketchBoost('bce')` — binary cross-entropy, подходит для multi-label
- `model.predict()` возвращает **raw logits**, нужен sigmoid для вероятностей
- `eval_sets` — список словарей `{'X': ..., 'y': ...}` для early stopping

In [8]:
mskf = MultilabelStratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_preds = np.zeros_like(y, dtype=np.float64)
test_preds = np.zeros((X_test.shape[0], len(target_cols)), dtype=np.float64)

for fold_idx, (train_idx, val_idx) in enumerate(mskf.split(X, y)):
    print(f'\n{"="*60}')
    print(f'FOLD {fold_idx + 1}/{N_FOLDS}')
    print(f'{"="*60}')
    
    X_tr, X_val = X[train_idx], X[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]
    
    # SketchBoost = GradientBoosting + RandomProjectionSketch(1) + use_hess=False
    model = SketchBoost('bce', **PARAMS)
    
    # fit принимает numpy, внутри конвертирует в CuPy на GPU
    model.fit(
        X_tr, y_tr,
        eval_sets=[{'X': X_val, 'y': y_val}],
    )
    
    # predict возвращает raw logits → sigmoid для вероятностей
    # (для ROC-AUC можно и без sigmoid, но для ансамбля лучше калиброванные)
    val_raw = model.predict(X_val)
    test_raw = model.predict(X_test)
    
    val_prob = 1.0 / (1.0 + np.exp(-val_raw))
    test_prob = 1.0 / (1.0 + np.exp(-test_raw))
    
    oof_preds[val_idx] = val_prob
    test_preds += test_prob / N_FOLDS
    
    # Fold score
    fold_aucs = []
    for t_idx, t_name in enumerate(target_cols):
        auc = roc_auc_score(y_val[:, t_idx], val_prob[:, t_idx])
        fold_aucs.append(auc)
    print(f'\nFold {fold_idx + 1} Macro AUC: {np.mean(fold_aucs):.5f}')
    
    # Освобождаем GPU память
    del model, X_tr, X_val, y_tr, y_val
    cp.get_default_memory_pool().free_all_blocks()
    gc.collect()


FOLD 1/5
[17:09:48] Stdout logging level is INFO.
[17:09:48] GDBT train starts. Max iter 5000, early stopping rounds 200
[17:11:20] Iter 0; Sample 0, BCE = 0.10413449111954776; 
[17:11:37] Iter 200; Sample 0, BCE = 0.08623143248942695; 
[17:11:54] Iter 400; Sample 0, BCE = 0.08510717302291537; 
[17:12:12] Iter 600; Sample 0, BCE = 0.0846255762983852; 
[17:12:31] Iter 800; Sample 0, BCE = 0.08440533801576201; 
[17:12:49] Iter 1000; Sample 0, BCE = 0.0842432997255599; 
[17:13:08] Iter 1200; Sample 0, BCE = 0.08414581562442408; 
[17:13:27] Iter 1400; Sample 0, BCE = 0.08406174152430103; 
[17:13:46] Iter 1600; Sample 0, BCE = 0.08400095817089098; 
[17:14:05] Iter 1800; Sample 0, BCE = 0.08394848835324963; 
[17:14:23] Iter 2000; Sample 0, BCE = 0.08390556918704095; 
[17:14:42] Iter 2200; Sample 0, BCE = 0.08386997240778442; 
[17:15:01] Iter 2400; Sample 0, BCE = 0.08384725855795781; 
[17:15:20] Iter 2600; Sample 0, BCE = 0.08382244054909785; 
[17:15:39] Iter 2800; Sample 0, BCE = 0.0838083

## 4. OOF скор

In [9]:
print('Per-target OOF AUC:')
per_target_auc = []
for t_idx, t_name in enumerate(target_cols):
    auc = roc_auc_score(y[:, t_idx], oof_preds[:, t_idx])
    per_target_auc.append(auc)
    print(f'  {t_name}: {auc:.5f}')

macro_auc = np.mean(per_target_auc)
print(f'\nOOF Macro AUC: {macro_auc:.5f}')

macro_auc_sklearn = roc_auc_score(y, oof_preds, average='macro')
print(f'OOF Macro AUC (sklearn): {macro_auc_sklearn:.5f}')

Per-target OOF AUC:
  target_1_1: 0.90621
  target_1_2: 0.81406
  target_1_3: 0.86720
  target_1_4: 0.82669
  target_1_5: 0.88037
  target_2_1: 0.81424
  target_2_2: 0.92691
  target_2_3: 0.79586
  target_2_4: 0.74659
  target_2_5: 0.77178
  target_2_6: 0.73893
  target_2_7: 0.83669
  target_2_8: 0.94279
  target_3_1: 0.68838
  target_3_2: 0.90687
  target_3_3: 0.74931
  target_3_4: 0.93540
  target_3_5: 0.97538
  target_4_1: 0.85136
  target_5_1: 0.75074
  target_5_2: 0.73619
  target_6_1: 0.72974
  target_6_2: 0.73106
  target_6_3: 0.76316
  target_6_4: 0.85381
  target_6_5: 0.91814
  target_7_1: 0.79949
  target_7_2: 0.84055
  target_7_3: 0.79203
  target_8_1: 0.97622
  target_8_2: 0.83372
  target_8_3: 0.87072
  target_9_1: 0.78460
  target_9_2: 0.82286
  target_9_3: 0.68791
  target_9_4: 0.90615
  target_9_5: 0.82892
  target_9_6: 0.68593
  target_9_7: 0.75082
  target_9_8: 0.92907
  target_10_1: 0.75408

OOF Macro AUC: 0.82246
OOF Macro AUC (sklearn): 0.82246


## 5. Сабмит

In [10]:
predict_cols = [t.replace('target_', 'predict_') for t in target_cols]

submit = pd.DataFrame({'customer_id': customer_ids_test})
for i, col in enumerate(predict_cols):
    submit[col] = test_preds[:, i]

sample = pd.read_parquet(f'{DATA_DIR}sample_submit.parquet')
assert list(submit.columns) == list(sample.columns), 'Column mismatch!'
assert len(submit) == len(sample), 'Row count mismatch!'

submit.to_parquet('submit_pyboost.parquet', index=False)
print(f'Saved: submit_pyboost.parquet, shape={submit.shape}')
submit.head()

Saved: submit_pyboost.parquet, shape=(250000, 42)


,customer_id,predict_1_1,predict_1_2,predict_1_3,predict_1_4,predict_1_5,predict_2_1,predict_2_2,predict_2_3,predict_2_4,...,predict_8_3,predict_9_1,predict_9_2,predict_9_3,predict_9_4,predict_9_5,predict_9_6,predict_9_7,predict_9_8,predict_10_1
0,1750001,0.500112,0.500535,0.502148,0.501752,0.500159,0.502026,0.500879,0.500084,0.502297,...,0.506126,0.500655,0.519282,0.504499,0.500008,0.500261,0.594166,0.523146,0.500325,0.593218
1,1750002,0.501004,0.501159,0.509280,0.502473,0.500399,0.508421,0.501976,0.500149,0.502487,...,0.501347,0.500366,0.554821,0.506564,0.503294,0.502575,0.581058,0.526511,0.500053,0.553236
2,1750003,0.501204,0.501398,0.504463,0.507762,0.500761,0.501573,0.501756,0.500376,0.500894,...,0.503347,0.501611,0.508442,0.503512,0.500009,0.500067,0.551635,0.507888,0.500063,0.601752
3,1750004,0.500049,0.500136,0.500461,0.500940,0.500004,0.501095,0.500332,0.500045,0.502646,...,0.506486,0.500376,0.509759,0.505116,0.500079,0.501529,0.586577,0.532274,0.500266,0.596284
4,1750005,0.503561,0.500683,0.505976,0.506045,0.500072,0.501486,0.501190,0.500185,0.500737,...,0.501885,0.500346,0.515177,0.505770,0.500011,0.502463,0.560481,0.526654,0.500133,0.585435


In [11]:
# OOF для будущего ансамблирования
oof_df = pd.DataFrame({'customer_id': customer_ids_train})
for i, col in enumerate(predict_cols):
    oof_df[col] = oof_preds[:, i]
oof_df.to_parquet('oof_pyboost.parquet', index=False)
print('Saved: oof_pyboost.parquet')

Saved: oof_pyboost.parquet


---

## TODO: Эксперименты для улучшения скора

### 1. Подбор гиперпараметров

Самые влиятельные параметры для перебора (по приоритету):

| Параметр | Диапазон | Почему важен |
|----------|----------|-------------|
| `lr` | 0.01, 0.03, 0.05, 0.08, 0.1 | Основной trade-off скорость/качество |
| `max_depth` | 4, 5, 6, 7, 8 | Сложность деревьев; глубже = больше переобучение |
| `lambda_l2` | 0.1, 0.5, 1, 3, 10 | Регуляризация; с малым lr можно ослабить |
| `subsample` | 0.5, 0.6, 0.7, 0.8, 0.9, 1.0 | Стохастичность по строкам |
| `colsample` | 0.5, 0.6, 0.7, 0.8, 0.9, 1.0 | Стохастичность по фичам |
| `min_data_in_leaf` | 1, 5, 10, 20, 50 | Регуляризация через мин. размер листа |

In [12]:
# TODO: Пример подбора lr и max_depth на одном фолде
# from itertools import product
#
# results = []
# for lr, depth in product([0.03, 0.05, 0.08], [5, 6, 7]):
#     model = SketchBoost('bce', ntrees=5000, lr=lr, max_depth=depth,
#                         es=200, verbose=0, lambda_l2=1,
#                         subsample=0.8, colsample=0.8)
#     model.fit(X_tr, y_tr, eval_sets=[{'X': X_val, 'y': y_val}])
#     pred = 1.0 / (1.0 + np.exp(-model.predict(X_val)))
#     score = roc_auc_score(y_val, pred, average='macro')
#     results.append({'lr': lr, 'depth': depth, 'auc': score})
#     print(f'lr={lr}, depth={depth} → AUC={score:.5f}')
#     del model; cp.get_default_memory_pool().free_all_blocks(); gc.collect()
#
# pd.DataFrame(results).sort_values('auc', ascending=False)

### 2. Стратегии скетчинга

SketchBoost по умолчанию использует `RandomProjectionSketch(1)`. Можно попробовать другие:

```python
from py_boost.multioutput.sketching import RandomProjectionSketch, RandomSamplingSketch

# Вариант 1: увеличить размер проекции (точнее, но медленнее)
sketch = RandomProjectionSketch(n_components=3)

# Вариант 2: случайная подвыборка таргетов (хорошо с use_hess=True)
sketch = RandomSamplingSketch(n_outputs=10)

# Вариант 3: топ выходов по градиенту (нужен cuml)
# from py_boost.multioutput.sketching import TopOutputsSketch
# sketch = TopOutputsSketch(n_outputs=10)

# Вариант 4: SVD-скетч (нужен cuml)
# from py_boost.multioutput.sketching import SVDSketch
# sketch = SVDSketch(n_components=3)

model = GradientBoosting(
    'bce', ntrees=5000, lr=0.05,
    multioutput_sketch=sketch,
    use_hess=True,   # с RandomSamplingSketch лучше включить
    ...
)
```

**Рекомендации из туториала:**
- `RandomProjectionSketch(1)` + `use_hess=False` — самый быстрый (дефолт SketchBoost)
- `RandomSamplingSketch(10)` + `use_hess=True` — может быть точнее на задачах с коррелированными таргетами
- Увеличение `n_components`/`n_outputs` — точнее, но медленнее

In [13]:
# TODO: Сравнение стратегий скетчинга на одном фолде
# configs = [
#     ('SketchBoost default',       SketchBoost('bce', **PARAMS)),
#     ('RandomProjection(3)',       GradientBoosting('bce', multioutput_sketch=RandomProjectionSketch(3), use_hess=False, **PARAMS)),
#     ('RandomSampling(10)+hess',   GradientBoosting('bce', multioutput_sketch=RandomSamplingSketch(10), use_hess=True, **{k:v for k,v in PARAMS.items() if k != 'use_hess'})),
#     ('No sketch (exact)',         GradientBoosting('bce', use_hess=True, **{k:v for k,v in PARAMS.items() if k != 'use_hess'})),
# ]
#
# import time
# for name, model in configs:
#     t0 = time.time()
#     model.fit(X_tr, y_tr, eval_sets=[{'X': X_val, 'y': y_val}])
#     elapsed = time.time() - t0
#     pred = 1.0 / (1.0 + np.exp(-model.predict(X_val)))
#     score = roc_auc_score(y_val, pred, average='macro')
#     print(f'{name:30s} → AUC={score:.5f}, time={elapsed:.0f}s')
#     del model; cp.get_default_memory_pool().free_all_blocks(); gc.collect()

### 3. use_hess и gd_steps

- `use_hess=True` — использует вторые производные (Гессиан) для оптимизации листовых значений. Медленнее, но может дать лучшее качество, особенно при дисбалансе классов (наш случай!)
- `gd_steps` — сколько шагов градиентного спуска делать в каждом листе. По умолчанию 1. Увеличение до 3-5 может помочь совместно с `use_hess=True`

```python
# Попробовать:
model = SketchBoost('bce', ntrees=5000, lr=0.03, use_hess=True, gd_steps=3, ...)
```

### 4. Кастомная функция потерь

Py-Boost позволяет писать свои loss-функции на CuPy. Для multi-label можно попробовать:
- **Focal loss** — фокусируется на сложных примерах, полезно при дисбалансе
- **Weighted BCE** — разный вес для разных таргетов

```python
from py_boost.gpu.losses import Loss
import cupy as cp

class FocalBCE(Loss):
    """Focal loss для binary cross-entropy: фокус на сложных примерах."""
    def __init__(self, gamma=2.0):
        self.gamma = gamma
    
    def base_score(self, y_true):
        p = y_true.mean(axis=0)
        return cp.log(p / (1 - p + 1e-7))
    
    def get_grad_hess(self, y_true, y_pred):
        p = 1.0 / (1.0 + cp.exp(-y_pred))
        pt = y_true * p + (1 - y_true) * (1 - p)
        focal_weight = (1 - pt) ** self.gamma
        grad = focal_weight * (p - y_true)
        hess = focal_weight * p * (1 - p)
        return grad, hess
    
    def postprocess_output(self, y_pred):
        return 1.0 / (1.0 + cp.exp(-y_pred))

# Использование:
# model = SketchBoost(FocalBCE(gamma=2.0), ntrees=5000, ...)
```

Подробнее: [Tutorial_3_Custom_features.ipynb](https://github.com/sb-ai-lab/Py-Boost/blob/master/tutorials/Tutorial_3_Custom_features.ipynb)

### 5. Другие идеи

- **sample_weight** — можно передать веса сэмплов в `model.fit(X, y, sample_weight=w)`. Например, увеличить вес редких классов
- **Extra features** — добавить фичей из extra features
- **ONNX export** — обученную модель можно сконвертировать для CPU-инференса через ONNX: [Tutorial_5](https://github.com/sb-ai-lab/Py-Boost/blob/master/tutorials/Tutorial_5_ONNX_inference.ipynb)
- **Ансамбль** — блендинг OOF предсказаний SketchBoost с CatBoost для diversity
- **Post-processing** — калибровка вероятностей (Platt scaling) после обучения

In [14]:
# TODO: Итоговый чеклист экспериментов
#
# [ ] lr: 0.03 vs 0.05 vs 0.08
# [ ] max_depth: 5 vs 6 vs 7
# [ ] use_hess=True vs False
# [ ] gd_steps: 1 vs 3
# [ ] RandomSamplingSketch(10) + use_hess=True
# [ ] RandomProjectionSketch(3) vs (1)
# [ ] subsample: 0.7 vs 0.8 vs 0.9
# [ ] lambda_l2: 0.5 vs 1 vs 3
# [ ] FocalBCE(gamma=2) vs стандартный BCE
# [ ] Добавить extra features
# [ ] Ансамбль с CatBoost